# C. elegans Annotation Viewer — Demo with Synthetic Data

Same workflow as `viewer.ipynb` but uses generated data so it runs
anywhere without access to the real image files.

In [1]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import napari
import numpy as np

from celegans_annotator import add_lineage_view
from celegans_annotator.clipping_planes import (
    add_clipping_plane_widgets,
    init_clipping_planes,
)

## Generate synthetic data

Create fake image volumes (random blobs) and a small C. elegans lineage
with known division patterns.

In [3]:
rng = np.random.default_rng(42)
n_timepoints = 8
vol_shape = (60, 80, 100)  # x, y, z spatial dims

# Generate two "channel" image stacks with gaussian blobs
green_data = np.zeros((n_timepoints, *vol_shape), dtype=np.float32)
red_data = np.zeros((n_timepoints, *vol_shape), dtype=np.float32)


def add_blob(vol, center, sigma=4, intensity=200):
    """Add a gaussian blob to a 3D volume."""
    zz, yy, xx = np.ogrid[0 : vol.shape[0], 0 : vol.shape[1], 0 : vol.shape[2]]
    dist_sq = (zz - center[0]) ** 2 + (yy - center[1]) ** 2 + (xx - center[2]) ** 2
    vol += (intensity * np.exp(-dist_sq / (2 * sigma**2))).astype(np.float32)


# Build a synthetic C. elegans-like lineage
# P0 -> AB + P1, AB -> ABa + ABp, P1 -> EMS + P2, EMS -> MS + E
nuclei_coords = []
nuclei_labels = []

# Cell positions: {name: (z, y, x)} — drift slightly each timepoint
lineage = {
    # (name, start_t, end_t, base_pos)
    ("P0", 0, 0, (30, 40, 50)),
    ("AB", 1, 2, (25, 35, 45)),
    ("P1", 1, 2, (35, 45, 55)),
    ("ABa", 3, 7, (20, 30, 40)),
    ("ABp", 3, 7, (25, 35, 50)),
    ("EMS", 3, 3, (35, 45, 55)),
    ("P2", 3, 7, (40, 50, 65)),
    ("MS", 4, 7, (32, 42, 52)),
    ("E", 4, 7, (38, 48, 58)),
}

for name, t_start, t_end, (bz, by, bx) in lineage:
    for t in range(t_start, t_end + 1):
        drift = rng.normal(0, 1, size=3)
        z = bz + drift[0] + (t - t_start) * 0.5
        y = by + drift[1] + (t - t_start) * 0.3
        x = bx + drift[2] + (t - t_start) * 0.2
        nuclei_coords.append([t, z, y, x])
        nuclei_labels.append(name)

        # Add blobs at each nucleus position in both channels
        center = (
            int(np.clip(z, 0, vol_shape[0] - 1)),
            int(np.clip(y, 0, vol_shape[1] - 1)),
            int(np.clip(x, 0, vol_shape[2] - 1)),
        )
        add_blob(green_data[t], center, sigma=4, intensity=200)
        add_blob(red_data[t], center, sigma=3, intensity=150)

# Add some background noise
green_data += rng.poisson(5, green_data.shape).astype(np.float32)
red_data += rng.poisson(3, red_data.shape).astype(np.float32)

nuclei_coords = np.array(nuclei_coords)
print(f"Created {len(nuclei_coords)} nuclei across {n_timepoints} timepoints")
print(f"Cell names: {sorted(set(nuclei_labels))}")

Created 29 nuclei across 8 timepoints
Cell names: ['AB', 'ABa', 'ABp', 'E', 'EMS', 'MS', 'P0', 'P1', 'P2']


## Create napari viewer

In [4]:
viewer = napari.Viewer()

viewer.add_image(
    green_data,
    blending="additive",
    contrast_limits=(0, 300),
    colormap="green",
    rendering="attenuated_mip",
    attenuation=0.75,
    name="green",
)
viewer.add_image(
    red_data,
    blending="additive",
    contrast_limits=(0, 300),
    colormap="red",
    rendering="attenuated_mip",
    attenuation=0.75,
    name="red",
)

viewer.add_points(
    nuclei_coords,
    ndim=4,
    opacity=0.7,
    size=6,
    face_color="transparent",
    border_color="cyan",
    properties={"name": nuclei_labels},
    text="name",
    blending="additive",
)

green_layer = viewer.layers["green"]
red_layer = viewer.layers["red"]
point_layer = viewer.layers["nuclei_coords"]

## Add clipping planes

In [5]:
t, x, y, z = green_data.shape
layers = [green_layer, red_layer, point_layer]

init_clipping_planes(layers, shape=(x, y, z))
add_clipping_plane_widgets(viewer, layers, shape=(x, y, z))

viewer.dims.ndisplay = 3

## Add lineage tree view

In [7]:
add_lineage_view(viewer, nuclei_coords, nuclei_labels)

In [8]:
napari.run()